In [1]:
import numpy as np
import faiss
import chromadb
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# -----------------------------
# 1. Load Dataset (arXiv-like)
# -----------------------------

dataset = load_dataset("ccdv/arxiv-summarization", split="train[:200]")

documents = [d["article"] for d in dataset]

/home/mahsalot/miniconda3/envs/llm-lora-qlora/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -----------------------------
# 2. Chunking
# -----------------------------
def chunk_text(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunks.append(" ".join(words[i:i+chunk_size]))
    return chunks

chunks = []
for doc in documents:
    chunks.extend(chunk_text(doc))

In [3]:
len(chunks)

15567

## Generate test queries using Mistral 7b and the chunks I already have

In [4]:
import torch
import json
import random
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

# =========================
# CONFIG
# =========================
model_name = "mistralai/Mistral-7B-Instruct-v0.3"

# =========================
# TOKENIZER
# =========================
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =========================
# MODEL
# =========================
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.eval()

print("Model loaded successfully!")

# # =========================
# # GENERATION FUNCTION
# # =========================
# def generate_text(prompt, max_new_tokens=128):
#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#     with torch.no_grad():
#         output = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             temperature=0.7,
#             top_p=0.9,
#             do_sample=True,
#             pad_token_id=tokenizer.eos_token_id
#         )

#     decoded = tokenizer.decode(output[0], skip_special_tokens=True)

#     # remove prompt part
#     return decoded[len(prompt):].strip()

# # =========================
# # CLEAN OUTPUT
# # =========================
# def clean_query(text):
#     return text.strip().split("\n")[0]

# # =========================
# # DATASET GENERATION
# # =========================
# def generate_synthetic_queries(chunks, num_queries=100):
#     dataset = []

#     sampled_chunks = random.sample(chunks, min(len(chunks), num_queries))

#     for i, chunk in tqdm(enumerate(sampled_chunks)):

#         prompt = f"""
# You are an AI research scientist.

# Generate ONE technical search query from the excerpt.

# Rules:
# - One sentence only
# - No explanations
# - Must be specific and technical
# - Do NOT mention the excerpt

# Excerpt:
# {chunk}

# Question:
# """

#         try:
#             raw = generate_text(prompt)
#             query = clean_query(raw)

#             dataset.append({
#                 "query_id": f"q_{i}",
#                 "query": query,
#                 "ground_truth_context": chunk
#             })

#         except Exception as e:
#             print(f"Error at {i}: {e}")

#         if i % 10 == 0:
#             print(f"Generated {i}/{num_queries}")

#     return dataset

# # =========================
# # SAVE DATASET
# # =========================
# def save_dataset(data, path="synthetic_queries.json"):
#     with open(path, "w") as f:
#         json.dump(data, f, indent=2)

# # =========================
# # MAIN
# # =========================
# if __name__ == "__main__":

#     print("Starting generation...\n")

#     test_dataset = generate_synthetic_queries(chunks, num_queries=1000)

#     save_dataset(test_dataset)

#     print("\nSample output:")
#     for item in test_dataset[:3]:
#         print(item)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████████████████████████████████████████████████████| 291/291 [00:01<00:00, 190.71it/s, Materializing param=model.norm.weight]


Model loaded successfully!


In [5]:
import json
import pandas as pd
with open("synthetic_queries.json", "r") as f:
    test_dataset = json.load(f)

test_df = pd.DataFrame(test_dataset)

In [6]:
test_df.shape


(1000, 3)

In [7]:
# -----------------------------
# 3. Embedding Model (STRONG)
# -----------------------------
from sentence_transformers import SentenceTransformer

# 1. Initialize the Qwen3-Embedding-8B model
# We use device_map="auto" to handle the 8B model size (16GB+ VRAM recommended)
# trust_remote_code=True is required for the Qwen architecture
embedder = SentenceTransformer(
    "Qwen/Qwen3-VL-Embedding-8B", 
    device="cuda", 
    trust_remote_code=True,
    model_kwargs={"torch_dtype": "bfloat16"}
)

# 2. Encode with Instructions
# Qwen3 is an "instruction-aware" model. For RAG chunks, you should 
# specify that these are documents/passages.
embeddings = embedder.encode(
    chunks,
    prompt_name="document", # Tells Qwen to treat these as stored knowledge
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=8 # Adjust based on your VRAM
)

Loading weights: 100%|█████████████████████████████████████████████████████████████████| 749/749 [00:00<00:00, 972.80it/s, Materializing param=visual.pos_embed.weight]
Default prompt name is set to 'default'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.
Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1946/1946 [17:18<00:00,  1.87it/s]


In [8]:
import chromadb
from chromadb.config import Settings

# 1. Create a Persistent Client (saves data to './chroma_db' folder)
client = chromadb.PersistentClient(path="./chroma_db")

# 2. Get or create the collection
collection = client.get_or_create_collection(name="arxiv_rag")

# 3. Safe Batching Logic
# We use a batch size of 2000 to be safely under the 5,461 limit
MAX_BATCH = 2000 

for i in range(0, len(chunks), MAX_BATCH):
    batch_end = min(i + MAX_BATCH, len(chunks))
    
    collection.add(
        documents=chunks[i:batch_end],
        embeddings=embeddings[i:batch_end].tolist(), # Convert NumPy to List
        ids=[str(j) for j in range(i, batch_end)]
    )
    print(f"Uploaded batch: {i} to {batch_end}")

Uploaded batch: 0 to 2000
Uploaded batch: 2000 to 4000
Uploaded batch: 4000 to 6000
Uploaded batch: 6000 to 8000
Uploaded batch: 8000 to 10000
Uploaded batch: 10000 to 12000
Uploaded batch: 12000 to 14000
Uploaded batch: 14000 to 15567


In [9]:
# -----------------------------
# 5. FAISS Setup
# -----------------------------
dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(embeddings)

In [ ]:
# -----------------------------
# 6. LLM (for answer generation)
# -----------------------------
# from openai import OpenAI

# client = OpenAI(api_key=".....")

# def generate_answer_gpt4(context, query):
#     response = client.chat.completions.create(
#         model="gpt-4o", # Or "gpt-4"
#         messages=[
#             {"role": "system", "content": "Answer the question based ONLY on the provided context."},
#             {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
#         ],
#         temperature=0 # Keep it 0 for factual research tasks
#     )
#     return response.choices[0].message.content


def generate_answer_mistral(context, query, max_new_tokens=128):
    messages = [
        {
            "role": "system",
            "content": "Answer ONLY using the provided context. If not in context, say 'I don't know.'"
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}"
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt"
    )


    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs.get("attention_mask", None)
    if attention_mask is not None:
        attention_mask = attention_mask.to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return decoded.split("assistant")[-1].strip()

In [18]:
# -----------------------------
# 7. Query Function
# -----------------------------
def retrieve(query, k=5):
    q_emb = embedder.encode(
        [query],
        prompt_name="query",   # IMPORTANT for Qwen symmetry
        normalize_embeddings=True
    )

    scores, idx = faiss_index.search(q_emb, k)
    return [chunks[i] for i in idx[0]]

In [19]:
queries = list(test_df["query"])

In [20]:
# -----------------------------
# 8. Evaluation Dataset (toy)
# -----------------------------

# Ground-truth: naive matching (for demo)
def is_relevant(chunk, query):
    keywords = query.lower().split()
    return any(k in chunk.lower() for k in keywords)

In [34]:
# -----------------------------
# 9. Evaluation: Recall@k, MRR
# -----------------------------
def evaluate_retrieval(k=5):
    recall_scores = []
    mrr_scores = []

    for q in queries:
        retrieved = retrieve(q, k=k)

        found = False
        rank = None

        for i, chunk in enumerate(retrieved):
            if is_relevant(chunk, q):
                found = True
                rank = i + 1
                break

        recall_scores.append(1 if found else 0)

        if found:
            mrr_scores.append(1 / rank)
        else:
            mrr_scores.append(0)

    print(f"Recall@{k}: {np.mean(recall_scores):.3f}")
    print(f"MRR: {np.mean(mrr_scores):.3f}")

# -----------------------------
# 10. End-to-End QA Evaluation
# -----------------------------
def evaluate_qa():
    answers = []
    for q in tqdm(queries):
        retrieved = retrieve(q, k=5)
        context = "\n".join(retrieved)
        answers.append(generate_answer_mistral(context, q))
    return answers


In [32]:
print("Evaluating retrieval...")
evaluate_retrieval(k=5)

Evaluating retrieval...
Recall@5: 0.998
MRR: 0.996


In [35]:
print("\nEvaluating QA...")
answers = evaluate_qa()


Evaluating QA...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [58:55<00:00,  3.54s/it]


In [36]:
answers[0]

'Answer ONLY using the provided context. If not in context, say \'I don\'t know.\'\n\nContext:\n. however , such caveats are not usually considered when using irdc data to constrain msf . thus we abstain from such considerations . our study suggests that many irdcs , if not most , are not related to msf . one thus has to be prudent when using irdc properties to constrain msf initial conditions . most studies discussing irdcs as pre - msf sites concentrated on very opaque irdcs of large angular size . these clouds often violate eq . ( [ eq : mass - size - limit ] ) , and many of them are good\napproximates an msf limit . many well - studied irdcs ( 25%50% ) fall short of this threshold ( section [ sec : well - studied - irdcs ] ) . less certain data for complete irdc samples suggests that most irdcs obey eq.([eq : mass - size - limit ] ) , and will thus not form massive stars ( section [ sec : typical - irdcs ] ) . still , most of the mass contained by irdcs might be in clouds forming m

In [37]:
len(answers)

1000